# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [162]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [163]:
#loading the dataset
df=pd.read_csv('data/AviationData.csv',encoding='latin-1')
df.info()

C:\Users\kirukuwaweru\AppData\Local\Temp\ipykernel_12392\23299201.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('data/AviationData.csv',encoding='latin-1')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [165]:
#simple data analysis
df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [166]:
cols = [
    'Total.Fatal.Injuries',
    'Total.Serious.Injuries',
    'Total.Minor.Injuries',
    'Total.Uninjured'
]
#fill NaN values with 0
df['Total_Passengers'] = df[cols].fillna(0).sum(axis=1)

total_passengers=df['Total.Fatal.Injuries'].sum()+df['Total.Serious.Injuries'].sum()+df['Total.Minor.Injuries'].sum()+df['Total.Uninjured'].sum()
total_fatal_injuries=df['Total.Fatal.Injuries'].sum()
fatality_rate=(total_fatal_injuries/total_passengers)*100

print(f"{fatality_rate:.2f}%",)

(df['Total_Passengers'] > 20).sum()

df.info()

9.28%
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 32 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 no

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [167]:
#check for duplicates
duplicates=df.duplicated().sum()
print('Duplicated entries:',duplicates)

#Removing non-airplane entries
airplane_df=df[df['Aircraft.Category']=='Airplane']

#removing amateur builds
professional_df=airplane_df[airplane_df['Amateur.Built']=='No']

#removing accidents > 40years ago
#drop null values in 'Event.Date' column
professional_df=professional_df.dropna(subset=['Event.Date'])
#convert 'Event.Date' to datetime format
professional_df['Event.Date']=pd.to_datetime(professional_df['Event.Date'])

#df['date'].dt.year
filtered_df=professional_df[professional_df['Event.Date'].dt.year>1983]

#contructor for destroyed column
filtered_df['Destroyed']=filtered_df['Aircraft.damage'].apply(lambda x: 1 if x== 'Destroyed' else 0)
filtered_df.info()



Duplicated entries: 0
<class 'pandas.core.frame.DataFrame'>
Index: 21437 entries, 7708 to 88886
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                21437 non-null  object        
 1   Investigation.Type      21437 non-null  object        
 2   Accident.Number         21437 non-null  object        
 3   Event.Date              21437 non-null  datetime64[ns]
 4   Location                21431 non-null  object        
 5   Country                 21436 non-null  object        
 6   Latitude                19169 non-null  object        
 7   Longitude               19163 non-null  object        
 8   Airport.Code            13978 non-null  object        
 9   Airport.Name            14065 non-null  object        
 10  Injury.Severity         20624 non-null  object        
 11  Aircraft.damage         20212 non-null  object        
 12  Aircraft.Category       21

C:\Users\kirukuwaweru\AppData\Local\Temp\ipykernel_12392\2860384730.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Destroyed']=filtered_df['Aircraft.damage'].apply(lambda x: 1 if x== 'Destroyed' else 0)


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [168]:
filtered_df.head()
# drop null 'MAKE' values
filtered_df=filtered_df.dropna(subset=['Make'])

#standardize 'Make' column to title case
filtered_df['Make']=filtered_df['Make'].str.title()

#dropping makes with less than 50 occurrences
#analysis section. Unique entries in 'Make' column and their counts
filtered_df['Make'].value_counts().head(50) 
new_df=filtered_df[filtered_df['Make'].map(filtered_df['Make'].value_counts())>50]
new_df['Make'].value_counts().head(50)
new_df.info()

# #check which 'Make' has the highest number of destroyed aircraft

# destroyed_df=new_df[['Make','Destroyed']]
# destroyed_df.head()

# destroyed_counts=destroyed_df.groupby('Make')['Destroyed'].sum().sort_values(ascending=False)

# destroyed_counts.info()






<class 'pandas.core.frame.DataFrame'>
Index: 17836 entries, 7708 to 88886
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                17836 non-null  object        
 1   Investigation.Type      17836 non-null  object        
 2   Accident.Number         17836 non-null  object        
 3   Event.Date              17836 non-null  datetime64[ns]
 4   Location                17832 non-null  object        
 5   Country                 17835 non-null  object        
 6   Latitude                15960 non-null  object        
 7   Longitude               15957 non-null  object        
 8   Airport.Code            11621 non-null  object        
 9   Airport.Name            11728 non-null  object        
 10  Injury.Severity         17130 non-null  object        
 11  Aircraft.damage         16787 non-null  object        
 12  Aircraft.Category       17836 non-null  object  

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [158]:
# remove nans in the model column

new_df = new_df.dropna(subset=['Model'])

# inspect the colum and check if model no. unique 

new_df[['Make','Model']].head(50)

new_df[['Model','Make']].nunique()
new_df.groupby('Make')['Model'].nunique().sort_values(ascending=False)# checking the number of unique models for each make

new_df['Plane Type']=new_df['Make']+'_'+new_df['Model']
new_df.info()



<class 'pandas.core.frame.DataFrame'>
Index: 17823 entries, 7708 to 88886
Data columns (total 34 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                17823 non-null  object        
 1   Investigation.Type      17823 non-null  object        
 2   Accident.Number         17823 non-null  object        
 3   Event.Date              17823 non-null  datetime64[ns]
 4   Location                17819 non-null  object        
 5   Country                 17822 non-null  object        
 6   Latitude                15951 non-null  object        
 7   Longitude               15948 non-null  object        
 8   Airport.Code            11615 non-null  object        
 9   Airport.Name            11720 non-null  object        
 10  Injury.Severity         17119 non-null  object        
 11  Aircraft.damage         16774 non-null  object        
 12  Aircraft.Category       17823 non-null  object  

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [169]:


#fill nans in the columns with 0s
new_df = new_df.fillna(0)
new_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17836 entries, 7708 to 88886
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                17836 non-null  object        
 1   Investigation.Type      17836 non-null  object        
 2   Accident.Number         17836 non-null  object        
 3   Event.Date              17836 non-null  datetime64[ns]
 4   Location                17836 non-null  object        
 5   Country                 17836 non-null  object        
 6   Latitude                17836 non-null  object        
 7   Longitude               17836 non-null  object        
 8   Airport.Code            17836 non-null  object        
 9   Airport.Name            17836 non-null  object        
 10  Injury.Severity         17836 non-null  object        
 11  Aircraft.damage         17836 non-null  object        
 12  Aircraft.Category       17836 non-null  object  

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [170]:
# remove columns that have too many nans and are not relevant to our analysis 

new_df.isnull().sum()

#drop columns with most nans and not relevant to our analysis
new_df = new_df.drop(columns=['Latitude','Longitude','Location','Airport.Code','Airport.Name','Schedule','Air.carrier'])
new_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 17836 entries, 7708 to 88886
Data columns (total 26 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Event.Id                17836 non-null  object        
 1   Investigation.Type      17836 non-null  object        
 2   Accident.Number         17836 non-null  object        
 3   Event.Date              17836 non-null  datetime64[ns]
 4   Country                 17836 non-null  object        
 5   Injury.Severity         17836 non-null  object        
 6   Aircraft.damage         17836 non-null  object        
 7   Aircraft.Category       17836 non-null  object        
 8   Registration.Number     17836 non-null  object        
 9   Make                    17836 non-null  object        
 10  Model                   17836 non-null  object        
 11  Amateur.Built           17836 non-null  object        
 12  Number.of.Engines       17836 non-null  float64 

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [171]:
cleaned_aviation_data = new_df.to_csv('data/cleanedAviationData.csv', index=False)